# End-to-End Columnar Ingestion Benchmark

This notebook benchmarks serialization plus serialized batch ingestion. It is intended to make serialization-driven gains visible rather than only measuring backend write throughput.

Each benchmark row reports `serialization_seconds`, `ingestion_seconds`, and `total_seconds`. Data generation, DB construction, and cleanup are intentionally outside the timed region.

## Imports and Availability

In [1]:
from __future__ import annotations

from dataclasses import dataclass
import importlib.util
import random
import shutil
import sys
import tempfile
import time
from pathlib import Path

repo_src = Path.cwd().parent / 'src'
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from gestaltdb import IndexMaintenanceMode
from gestaltdb.graphdb import Edge, GraphDB, GraphEntityDictSerializer, Node
from gestaltdb.kvstores import LevelDBStore, PyRexStore
from gestaltdb.serializers import JSONSerializer, PickleSerializer

HAS_LEVELDB = importlib.util.find_spec('plyvel') is not None
HAS_PYREX = importlib.util.find_spec('pyrex') is not None
HAS_PYARROW = importlib.util.find_spec('pyarrow') is not None
HAS_POLARS = importlib.util.find_spec('polars') is not None

print({
    'leveldb': HAS_LEVELDB,
    'pyrex_rocksdb': HAS_PYREX,
    'pyarrow': HAS_PYARROW,
    'polars': HAS_POLARS,
})

{'leveldb': True, 'pyrex_rocksdb': True, 'pyarrow': True, 'polars': True}


## Parameters

Edit these lists to sweep different graph sizes and batch sizes. Keep the defaults small for laptop runs; increase them for stable throughput numbers.

In [19]:
NODE_SIZES = [100_000]
EDGE_SIZES = [500_000]
BATCH_SIZES = [1_000, 10_000, 50_000]
SEED = 42
EDGE_TYPES = ['drug-to-protein', 'protein-to-disease', 'drug-to-disease']

# RocksDB settings used by PyRexStore. The defaults favor ingestion benchmarking.
ROCKSDB_DISABLE_WAL = False
ROCKSDB_PARALLELISM = 4
ROCKSDB_MAX_BACKGROUND_JOBS = 4
ROCKSDB_WRITE_BUFFER_SIZE = 64 * 1024 * 1024
ROCKSDB_BLOOM_BITS_PER_KEY = 8

## Dataset and Serialization Helpers

In [20]:
@dataclass(frozen=True)
class Dataset:
    node_ids: list[str]
    edge_ids: list[str]
    sources: list[str]
    targets: list[str]
    edge_types: list[str]
    node_kinds: list[str]
    node_groups: list[int]
    edge_weights: list[int]
    nodes: list[Node]
    edges: list[Edge]


def timed(func):
    start = time.perf_counter()
    result = func()
    return result, time.perf_counter() - start


def make_dataset(num_nodes: int, num_edges: int, seed: int) -> Dataset:
    rng = random.Random(seed)
    node_ids = [f'n{idx}' for idx in range(num_nodes)]
    node_kinds = ['entity'] * num_nodes
    node_groups = [idx % 10 for idx in range(num_nodes)]
    nodes = [
        Node(node_id=node_id, labels=['Entity'], properties={'kind': kind, 'group': group})
        for node_id, kind, group in zip(node_ids, node_kinds, node_groups)
    ]

    edge_ids = [f'e{idx}' for idx in range(num_edges)]
    sources = [f'n{rng.randrange(num_nodes)}' for _ in range(num_edges)]
    targets = [f'n{rng.randrange(num_nodes)}' for _ in range(num_edges)]
    edge_types = [EDGE_TYPES[idx % len(EDGE_TYPES)] for idx in range(num_edges)]
    edge_weights = [idx % 100 for idx in range(num_edges)]
    edges = [
        Edge(edge_id=edge_id, source=source, target=target, properties={'type': edge_type, 'weight': weight})
        for edge_id, source, target, edge_type, weight in zip(edge_ids, sources, targets, edge_types, edge_weights)
    ]

    return Dataset(
        node_ids=node_ids,
        edge_ids=edge_ids,
        sources=sources,
        targets=targets,
        edge_types=edge_types,
        node_kinds=node_kinds,
        node_groups=node_groups,
        edge_weights=edge_weights,
        nodes=nodes,
        edges=edges,
    )


def serialize_python_entities(dataset: Dataset, serializer_cls):
    entity_serializer = GraphEntityDictSerializer(serializer_cls())
    node_values = [entity_serializer.serialize(node, 'Node') for node in dataset.nodes]
    edge_values = [entity_serializer.serialize(edge, 'Edge') for edge in dataset.edges]
    return {
        'node_ids': dataset.node_ids,
        'node_values': node_values,
        'edge_ids': dataset.edge_ids,
        'sources': dataset.sources,
        'targets': dataset.targets,
        'edge_types': dataset.edge_types,
        'edge_values': edge_values,
    }


def serialize_json_entities_with_polars(dataset: Dataset):
    if not HAS_POLARS or not HAS_PYARROW:
        raise RuntimeError('Polars JSON serialization requires polars and pyarrow')
    import polars as pl
    import pyarrow as pa

    node_df = pl.DataFrame({
        'node_id': dataset.node_ids,
        'labels': [['Entity'] for _ in dataset.node_ids],
        'kind': dataset.node_kinds,
        'group': dataset.node_groups,
    })
    edge_df = pl.DataFrame({
        'edge_id': dataset.edge_ids,
        'source': dataset.sources,
        'target': dataset.targets,
        'edge_type': dataset.edge_types,
        'weight': dataset.edge_weights,
    })

    node_payloads = node_df.select(
        pl.struct([
            pl.col('node_id').alias('id'),
            pl.struct(['kind', 'group']).alias('properties'),
            pl.col('labels'),
        ]).struct.json_encode().alias('node_value')
    )['node_value'].to_arrow().cast(pa.binary())
    edge_payloads = edge_df.select(
        pl.struct([
            pl.col('edge_id').alias('id'),
            pl.col('source'),
            pl.col('target'),
            pl.struct([pl.col('edge_type').alias('type'), pl.col('weight')]).alias('properties'),
        ]).struct.json_encode().alias('edge_value')
    )['edge_value'].to_arrow().cast(pa.binary())

    return {
        'node_ids': node_df['node_id'].to_arrow(),
        'node_values': node_payloads,
        'edge_ids': edge_df['edge_id'].to_arrow(),
        'sources': edge_df['source'].to_arrow(),
        'targets': edge_df['target'].to_arrow(),
        'edge_types': edge_df['edge_type'].to_arrow(),
        'edge_values': edge_payloads,
    }

## Backend and Benchmark Functions

In [21]:
def open_graph(backend: str, path: str, serializer_cls):
    if backend == 'leveldb':
        return GraphDB(LevelDBStore(path=path), serializer_cls())
    if backend == 'rocksdb':
        return GraphDB(
            PyRexStore(
                path=path,
                parallelism=ROCKSDB_PARALLELISM,
                max_background_jobs=ROCKSDB_MAX_BACKGROUND_JOBS,
                write_buffer_size=ROCKSDB_WRITE_BUFFER_SIZE,
                bloom_bits_per_key=ROCKSDB_BLOOM_BITS_PER_KEY,
                disable_wal=ROCKSDB_DISABLE_WAL,
            ),
            serializer_cls(),
        )
    raise ValueError(f'unknown backend: {backend}')


def backend_available(backend: str) -> bool:
    return (backend == 'leveldb' and HAS_LEVELDB) or (backend == 'rocksdb' and HAS_PYREX)


def run_serialized_ingestion(graph: GraphDB, payloads: dict, batch_size: int):
    graph.ingest_nodes_arrow(payloads['node_ids'], payloads['node_values'], chunk_size=batch_size)
    graph.ingest_edges_arrow(
        payloads['edge_ids'],
        payloads['sources'],
        payloads['targets'],
        payloads['edge_types'],
        payloads['edge_values'],
        append_only=True,
        chunk_size=batch_size,
    )


def make_polars_entity_frames(dataset: Dataset):
    import polars as pl

    node_df = pl.DataFrame({
        'node_id': dataset.node_ids,
        'labels': [['Entity'] for _ in dataset.node_ids],
        'kind': dataset.node_kinds,
        'group': dataset.node_groups,
    })
    edge_df = pl.DataFrame({
        'edge_id': dataset.edge_ids,
        'source': dataset.sources,
        'target': dataset.targets,
        'edge_type': dataset.edge_types,
        'weight': dataset.edge_weights,
    })
    return node_df, edge_df


def run_high_level_polars_defer_rebuild(graph: GraphDB, dataset: Dataset, batch_size: int):
    node_df, edge_df = make_polars_entity_frames(dataset)
    return graph.ingest_polars(
        node_df,
        edge_df,
        node_property_columns=['kind', 'group'],
        edge_property_columns=['weight'],
        index_mode=IndexMaintenanceMode.DEFER_REBUILD,
        chunk_size=batch_size,
    )


def validate_ingestion(graph: GraphDB, dataset: Dataset):
    first_node_id = dataset.node_ids[0].encode('utf-8')
    first_edge_id = dataset.edge_ids[0].encode('utf-8')
    assert graph.store.get_node(first_node_id) is not None
    assert graph.store.get_edge(first_edge_id) is not None
    assert graph.neighbors_by_edge_type(dataset.sources[0], dataset.edge_types[0], direction='out')


BENCHMARK_SPECS = [
    {
        'name': 'leveldb-pickle-python-serialization',
        'backend': 'leveldb',
        'serializer': 'pickle',
        'serializer_cls': PickleSerializer,
        'serialization_path': 'python objects + PickleSerializer',
        'serialize': lambda dataset: serialize_python_entities(dataset, PickleSerializer),
    },
    {
        'name': 'leveldb-json-python-serialization',
        'backend': 'leveldb',
        'serializer': 'json',
        'serializer_cls': JSONSerializer,
        'serialization_path': 'python objects + JSONSerializer',
        'serialize': lambda dataset: serialize_python_entities(dataset, JSONSerializer),
    },
    {
        'name': 'rocksdb-pickle-python-serialization',
        'backend': 'rocksdb',
        'serializer': 'pickle',
        'serializer_cls': PickleSerializer,
        'serialization_path': 'python objects + PickleSerializer',
        'serialize': lambda dataset: serialize_python_entities(dataset, PickleSerializer),
    },
    {
        'name': 'rocksdb-json-python-serialization',
        'backend': 'rocksdb',
        'serializer': 'json',
        'serializer_cls': JSONSerializer,
        'serialization_path': 'python objects + JSONSerializer',
        'serialize': lambda dataset: serialize_python_entities(dataset, JSONSerializer),
    },
    {
        'name': 'rocksdb-json-polars-serialization',
        'backend': 'rocksdb',
        'serializer': 'json',
        'serializer_cls': JSONSerializer,
        'serialization_path': 'Polars struct.json_encode + Arrow binary payloads',
        'serialize': serialize_json_entities_with_polars,
        'requires_polars': True,
    },
    {
        'name': 'rocksdb-json-polars-high-level-defer-rebuild',
        'backend': 'rocksdb',
        'serializer': 'json',
        'serializer_cls': JSONSerializer,
        'serialization_path': 'GraphDB.ingest_polars entity columns + deferred one-pass rebuild',
        'high_level_polars_defer_rebuild': True,
        'requires_polars': True,
    },
]


def spec_available(spec: dict) -> bool:
    if not backend_available(spec['backend']):
        return False
    if spec.get('requires_polars') and not (HAS_POLARS and HAS_PYARROW):
        return False
    return True


def run_one_benchmark(spec: dict, dataset: Dataset, num_nodes: int, num_edges: int, batch_size: int):
    if not spec_available(spec):
        return {
            'benchmark': spec['name'],
            'backend': spec['backend'],
            'serializer': spec['serializer'],
            'serialization_path': spec['serialization_path'],
            'nodes': num_nodes,
            'edges': num_edges,
            'batch_size': batch_size,
            'status': 'skipped: missing optional dependency',
        }

    path = tempfile.mkdtemp(prefix=f"gestaltdb_{spec['backend']}_{spec['serializer']}_e2e_")
    graph = open_graph(spec['backend'], path, spec['serializer_cls'])
    try:
        native_columnar = bool(getattr(graph.store, 'has_native_columnar_ingestion', lambda: False)())
        if spec.get('high_level_polars_defer_rebuild'):
            graph.create_node_property_index('kind')
            graph.create_node_property_index('group')
            graph.create_edge_property_index('weight')
            serialization_seconds = 0.0
            _, ingestion_seconds = timed(lambda: run_high_level_polars_defer_rebuild(graph, dataset, batch_size))
        else:
            payloads, serialization_seconds = timed(lambda: spec['serialize'](dataset))
            _, ingestion_seconds = timed(lambda: run_serialized_ingestion(graph, payloads, batch_size))
        validate_ingestion(graph, dataset)
    finally:
        graph.close()
        shutil.rmtree(path, ignore_errors=True)

    total_seconds = serialization_seconds + ingestion_seconds
    return {
        'benchmark': spec['name'],
        'backend': spec['backend'],
        'serializer': spec['serializer'],
        'serialization_path': spec['serialization_path'],
        'ingestion_path': 'GraphDB.ingest_polars defer+rebuild' if spec.get('high_level_polars_defer_rebuild') else 'GraphDB.ingest_*_arrow serialized batch ingestion',
        'native_columnar': native_columnar,
        'nodes': num_nodes,
        'edges': num_edges,
        'batch_size': batch_size,
        'serialization_seconds': serialization_seconds,
        'ingestion_seconds': ingestion_seconds,
        'total_seconds': total_seconds,
        'serialization_share': serialization_seconds / total_seconds if total_seconds else None,
        'node_rate_total': num_nodes / total_seconds if total_seconds else None,
        'edge_rate_total': num_edges / total_seconds if total_seconds else None,
        'status': 'ok',
    }

## Run Parameter Grid

This runs every benchmark spec for each `(nodes, edges, batch_size)` combination. For larger sweeps, start with one backend and one batch size, then expand.

In [22]:
results = []
for num_nodes in NODE_SIZES:
    for num_edges in EDGE_SIZES:
        dataset = make_dataset(num_nodes, num_edges, SEED)
        for batch_size in BATCH_SIZES:
            for spec in BENCHMARK_SPECS:
                print(f"running {spec['name']} nodes={num_nodes:,} edges={num_edges:,} batch={batch_size:,}")
                result = run_one_benchmark(spec, dataset, num_nodes, num_edges, batch_size)
                print(result['status'])
                results.append(result)

len(results)

running leveldb-pickle-python-serialization nodes=100,000 edges=500,000 batch=1,000
ok
running leveldb-json-python-serialization nodes=100,000 edges=500,000 batch=1,000
ok
running rocksdb-pickle-python-serialization nodes=100,000 edges=500,000 batch=1,000
ok
running rocksdb-json-python-serialization nodes=100,000 edges=500,000 batch=1,000
ok
running rocksdb-json-polars-serialization nodes=100,000 edges=500,000 batch=1,000
ok
running leveldb-pickle-python-serialization nodes=100,000 edges=500,000 batch=10,000
ok
running leveldb-json-python-serialization nodes=100,000 edges=500,000 batch=10,000
ok
running rocksdb-pickle-python-serialization nodes=100,000 edges=500,000 batch=10,000
ok
running rocksdb-json-python-serialization nodes=100,000 edges=500,000 batch=10,000
ok
running rocksdb-json-polars-serialization nodes=100,000 edges=500,000 batch=10,000
ok
running leveldb-pickle-python-serialization nodes=100,000 edges=500,000 batch=50,000
ok
running leveldb-json-python-serialization nodes=1

15

## Final Results Matrix

The matrix makes serialization explicit. Use `serialization_share` to see whether the newest serialization path is materially reducing end-to-end cost, and use `ingestion_seconds` to separate that from backend write performance.

In [23]:
try:
    import polars as pl

    results_df = pl.DataFrame(results)
    display(
        results_df.select([
            'status',
            'benchmark',
            'backend',
            'serializer',
            'serialization_path',
            'native_columnar',
            'nodes',
            'edges',
            'batch_size',
            'serialization_seconds',
            'ingestion_seconds',
            'total_seconds',
            'serialization_share',
            'edge_rate_total',
        ]).sort(['nodes', 'edges', 'batch_size', 'backend', 'serializer', 'benchmark'])
    )
except Exception:
    results

status,benchmark,backend,serializer,serialization_path,native_columnar,nodes,edges,batch_size,serialization_seconds,ingestion_seconds,total_seconds,serialization_share,edge_rate_total
str,str,str,str,str,bool,i64,i64,i64,f64,f64,f64,f64,f64
"""ok""","""leveldb-json-python-serializat…","""leveldb""","""json""","""python objects + JSONSerialize…",false,100000,500000,1000,0.923685,2.900443,3.824128,0.241541,130748.762402
"""ok""","""leveldb-pickle-python-serializ…","""leveldb""","""pickle""","""python objects + PickleSeriali…",false,100000,500000,1000,0.291639,2.859077,3.150716,0.092563,158694.100106
"""ok""","""rocksdb-json-polars-serializat…","""rocksdb""","""json""","""Polars struct.json_encode + Ar…",true,100000,500000,1000,0.414709,2.395071,2.809779,0.147595,177949.912002
"""ok""","""rocksdb-json-python-serializat…","""rocksdb""","""json""","""python objects + JSONSerialize…",true,100000,500000,1000,0.911053,2.422532,3.333585,0.273295,149988.684719
"""ok""","""rocksdb-pickle-python-serializ…","""rocksdb""","""pickle""","""python objects + PickleSeriali…",true,100000,500000,1000,0.293358,2.324335,2.617693,0.112067,191007.853013
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ok""","""leveldb-json-python-serializat…","""leveldb""","""json""","""python objects + JSONSerialize…",false,100000,500000,50000,0.941068,2.452766,3.393834,0.277287,147326.008432
"""ok""","""leveldb-pickle-python-serializ…","""leveldb""","""pickle""","""python objects + PickleSeriali…",false,100000,500000,50000,0.293298,2.770409,3.063707,0.095733,163200.975043
"""ok""","""rocksdb-json-polars-serializat…","""rocksdb""","""json""","""Polars struct.json_encode + Ar…",true,100000,500000,50000,0.486856,2.355243,2.842099,0.171302,175926.313058


In [24]:
# !uv pip install numpy pandas

In [25]:
d = results_df.to_pandas()
d

,benchmark,backend,serializer,serialization_path,ingestion_path,native_columnar,nodes,edges,batch_size,serialization_seconds,ingestion_seconds,total_seconds,serialization_share,node_rate_total,edge_rate_total,status
0,leveldb-pickle-python-serialization,leveldb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,1000,0.291639,2.859077,3.150716,0.092563,31738.820021,158694.100106,ok
1,leveldb-json-python-serialization,leveldb,json,python objects + JSONSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,1000,0.923685,2.900443,3.824128,0.241541,26149.752480,130748.762402,ok
2,rocksdb-pickle-python-serialization,rocksdb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,1000,0.293358,2.324335,2.617693,0.112067,38201.570603,191007.853013,ok
3,rocksdb-json-python-serialization,rocksdb,json,python objects + JSONSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,1000,0.911053,2.422532,3.333585,0.273295,29997.736944,149988.684719,ok
4,rocksdb-json-polars-serialization,rocksdb,json,Polars struct.json_encode + Arrow binary payloads,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,1000,0.414709,2.395071,2.809779,0.147595,35589.982400,177949.912002,ok
5,leveldb-pickle-python-serialization,leveldb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,10000,0.296108,2.670838,2.966947,0.099802,33704.683946,168523.419731,ok
6,leveldb-json-python-serialization,leveldb,json,python objects + JSONSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,10000,0.903574,2.933382,3.836956,0.235492,26062.323203,130311.616013,ok
7,rocksdb-pickle-python-serialization,rocksdb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,10000,0.289211,2.339681,2.628892,0.110012,38038.841839,190194.209194,ok
8,rocksdb-json-python-serialization,rocksdb,json,python objects + JSONSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,10000,0.908462,2.471322,3.379784,0.268793,29587.689864,147938.449319,ok
9,rocksdb-json-polars-serialization,rocksdb,json,Polars struct.json_encode + Arrow binary payloads,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,10000,0.490075,2.370341,2.860416,0.171330,34959.950603,174799.753013,ok


In [26]:
d.sort_values('total_seconds', ascending = True)

,benchmark,backend,serializer,serialization_path,ingestion_path,native_columnar,nodes,edges,batch_size,serialization_seconds,ingestion_seconds,total_seconds,serialization_share,node_rate_total,edge_rate_total,status
2,rocksdb-pickle-python-serialization,rocksdb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,1000,0.293358,2.324335,2.617693,0.112067,38201.570603,191007.853013,ok
7,rocksdb-pickle-python-serialization,rocksdb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,10000,0.289211,2.339681,2.628892,0.110012,38038.841839,190194.209194,ok
12,rocksdb-pickle-python-serialization,rocksdb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,50000,0.292045,2.370242,2.662287,0.109697,37561.690212,187808.451060,ok
4,rocksdb-json-polars-serialization,rocksdb,json,Polars struct.json_encode + Arrow binary payloads,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,1000,0.414709,2.395071,2.809779,0.147595,35589.982400,177949.912002,ok
14,rocksdb-json-polars-serialization,rocksdb,json,Polars struct.json_encode + Arrow binary payloads,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,50000,0.486856,2.355243,2.842099,0.171302,35185.262612,175926.313058,ok
9,rocksdb-json-polars-serialization,rocksdb,json,Polars struct.json_encode + Arrow binary payloads,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,10000,0.490075,2.370341,2.860416,0.171330,34959.950603,174799.753013,ok
5,leveldb-pickle-python-serialization,leveldb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,10000,0.296108,2.670838,2.966947,0.099802,33704.683946,168523.419731,ok
10,leveldb-pickle-python-serialization,leveldb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,50000,0.293298,2.770409,3.063707,0.095733,32640.195009,163200.975043,ok
0,leveldb-pickle-python-serialization,leveldb,pickle,python objects + PickleSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,False,100000,500000,1000,0.291639,2.859077,3.150716,0.092563,31738.820021,158694.100106,ok
13,rocksdb-json-python-serialization,rocksdb,json,python objects + JSONSerializer,GraphDB.ingest_*_arrow serialized batch ingestion,True,100000,500000,50000,0.898263,2.403288,3.301551,0.272073,30288.792093,151443.960463,ok


## Interpreting the Matrix

- `serialization_seconds` is the time to produce `node_value` and `edge_value` payload columns from entity data.
- `ingestion_seconds` is the time to ingest those serialized payloads with `GraphDB.ingest_nodes_arrow` and `GraphDB.ingest_edges_arrow`.
- `total_seconds` is the end-to-end benchmark number for serialization plus ingestion.
- `rocksdb-json-polars-serialization` is the main benchmark for serialization gains: it uses Polars `struct.json_encode()` to build JSON payloads and then uses Arrow binary payload columns for ingestion.
- `native_columnar=True` means the PyRex/RocksDB runtime exposes `write_columnar_batch`; otherwise the same public API falls back to Python bulk writes.
- The benchmark intentionally uses append-only edge ingestion, matching the current columnar ingestion API.